In [1]:
import torch
import torch.nn as nn

In [2]:
batch_size = 64
sequence_len = 64
embedding_size = 32
head_size = 256
steps = 5000

In [3]:
# input & output initialization
with open(file="inputs-shakespeare.txt", mode="r", encoding="utf-8") as f:
    data = f.read()

chars = sorted(set(data))
vocab_size = len(chars)
ctoi = {char: i for i, char in enumerate(iterable=chars)}
itoc = dict(enumerate(iterable=chars))

split = int(len(data) * 0.9)
train, val = data[:split], data[split:]

x_train = torch.tensor([ctoi[ch] for ch in train[:-1]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=1)
y_train = torch.tensor([ctoi[ch] for ch in train[1:]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=1)

x_val = torch.tensor([ctoi[ch] for ch in val[:-1]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=sequence_len)
y_val = torch.tensor([ctoi[ch] for ch in val[1:]], dtype=torch.long).unfold(dimension=0, size=sequence_len, step=sequence_len)

In [4]:
# module declaration
class CausalMean(nn.Module):
    def __init__(self):
        super().__init__()
        tril = torch.tril(input=torch.ones(size=(sequence_len, sequence_len), dtype=torch.bool))
        self.register_buffer(name="mask", tensor=tril, persistent=False)

    def forward(self, input):
        T = input.size(dim=1)
        scores = torch.zeros(size=(T, T))
        scores = scores.masked_fill(mask=~self.mask[:T, :T], value=float("-inf"))
        weights = torch.softmax(input=scores, dim=-1)
        return weights @ input

In [5]:
# model declaration
class BagOfWords(nn.Module):
    def __init__(self):
        super().__init__()
        self.head = nn.Sequential(
            nn.Embedding(num_embeddings=vocab_size, embedding_dim=embedding_size),
            CausalMean(),
            nn.Linear(in_features=embedding_size, out_features=head_size),
            nn.ReLU(),
            nn.Linear(in_features=head_size, out_features=vocab_size),
        )

    def forward(self, input):
        return self.head(input=input)

In [6]:
# model initialization
model = BagOfWords()
# optimizer initialization
optimizer = torch.optim.AdamW(params=model.parameters(), lr=3e-3)

In [7]:
# training
for step in range(steps):
    optimizer.zero_grad()
    
    batch = torch.randint(low=0, high=len(x_train), size=(batch_size,))
    train_logits = model(input=x_train[batch])
    train_loss = nn.functional.cross_entropy(input=train_logits.view(-1, vocab_size), target=y_train[batch].view(-1))

    if step % 1000 == 0 or step == steps - 1:
        with torch.no_grad():
            val_logits = model(input=x_val)
            val_loss = nn.functional.cross_entropy(input=val_logits.view(-1, vocab_size), target=y_val.view(-1))
            print(f"Step {step}: train loss = {train_loss.item()}, val loss = {val_loss.item()}")

    train_loss.backward()
    optimizer.step()

Step 0: train loss = 4.169808864593506, val loss = 4.169455051422119


Step 1000: train loss = 3.019969940185547, val loss = 3.0677504539489746


Step 2000: train loss = 2.9949331283569336, val loss = 3.0376946926116943


Step 3000: train loss = 3.002302885055542, val loss = 3.030390977859497


Step 4000: train loss = 2.970529079437256, val loss = 3.0135393142700195


Step 4999: train loss = 2.948000192642212, val loss = 3.015688180923462


In [8]:
# prediction
with torch.inference_mode():
    output = ['R', 'O', 'M', 'E', 'O', ':', '\n']
    max_output_length = 1000

    i = torch.tensor([[ctoi[ch] for ch in output]], dtype=torch.long)            

    while len(output) < max_output_length:
        logits = model(input=i)

        P = torch.softmax(input=logits[0, -1, :], dim=0)
        j = torch.multinomial(input=P, num_samples=1, replacement=True).item()
        output.append(itoc[j])

        i = torch.tensor([[ctoi[ch] for ch in output[-sequence_len:]]], dtype=torch.long)

In [9]:
print(''.join(output))

ROMEO:

Alosdn,  paryytuues  isf r; ouiltdyr,gi?,
 inwlr m nonDar t ,hCsgeegoiorcoorsfW,
stee  NHt ueOOeo,c, osorrtwht ,m a aeo.seo nkl  uptwro foysohh lho
l umu:
iyclnaMuAl
vWrselarssvha  o eebRooddh EnE unAsgdy  ponvmnl iel E yern tiaGdyredcon d lr,a wo   bq?gihos a
w
ehdogwuato
oHlguTo n ke rTY:tphuicl:e saaael  mdh.hrnm'
db  u nsyvnWavyn i eebsase  diuom',oont tteodwoirgowm eHsa 
krwn ,uh tnif  c p ofh' fthrotera ne  em domoChaaco stcbsoo iosou :iermhurr iurspwd
ent tr vtisenrotn  e
 eu beapweeHt a
Afnrfi  wTiyhghoyorome rhisal stnhsd snl mccrdu wo iR Iartv eil i:orelo, f
itcsnnedr rfaehyuipoyd s
rlleeyssnitnen 
lteghnA aoi iurdgwa ote    t nHt rpognceoinlrat euda rCi lyuyhbe,sa onks ad uo tw mbootoy hhy  nu u
 dbeevll edrmwwnles au  
sht TrSmtyytngspoeru  sIlo ahogesouto d eiholtivhtterne,d dseco omm
 !. 
 F- fs  tA'pgeoggw:u
RroaeuKliu
QeENOEI i
NM CsrLcYsE H:eatuolrsrull  :u f

suasAr uSttn wroehyb eo,ai atoer od legonnc nyhele ogsc!airdrenr !
a  oopn,a
m'BttnWpmg ts gs-tnya t; 